# LCABrick with `snn_Backend` + Spike Post-Processing

This notebook runs `LCABrick` using `snn_Backend` (not `slca_Backend`) and then post-processes raw spike output to recover sparse coefficients.

Goal: produce a decoded sparse code that matches the expected CLASSO solution.

In [ ]:

import numpy as np
import pandas as pd

from fugu import Scaffold
from fugu.bricks import LCABrick
from fugu.backends import snn_Backend
from fugu.utils.export_utils import fill_results_from_graph

## Problem Setup (same spirit as LCA tutorial)

In [2]:
def normalize_columns(A: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(A, axis=0, keepdims=True) + 1e-12
    return A / norms

def classo_fista_nonneg(Phi, s, lam, max_iter=5000, tol=1e-9):
    """FISTA solver for nonnegative CLASSO: min ||s - Phi a||^2/2 + lam||a||_1."""
    Phi = normalize_columns(Phi)
    PhiT = Phi.T
    PhiTPhi = PhiT @ Phi
    PhiTs = PhiT @ s
    L = np.linalg.eigvalsh(PhiTPhi).max() + 1e-12
    tstep = 1.0 / L

    def prox_nonneg_l1(x, threshold):
        return np.maximum(0.0, x - threshold)

    N = Phi.shape[1]
    a = np.zeros(N)
    y_iter = a.copy()
    theta = 1.0

    for _ in range(max_iter):
        grad = PhiTPhi @ y_iter - PhiTs
        a_next = prox_nonneg_l1(y_iter - tstep * grad, lam * tstep)
        theta_next = 0.5 * (1 + np.sqrt(1 + 4 * theta * theta))
        y_iter = a_next + (theta - 1) / theta_next * (a_next - a)
        if np.linalg.norm(a_next - a) < tol * (np.linalg.norm(a) + 1e-12):
            a = a_next
            break
        a, theta = a_next, theta_next

    return a

# Dictionary and signal (same reference dictionary used in tests/tutorial context)
Phi = np.array([
    [0.3313, 0.8148, 0.4364],
    [0.8835, 0.3621, 0.2182],
    [0.3313, 0.4527, 0.8729],
], dtype=float)
# Phi = normalize_columns(Phi)

y = np.array([0.5, 1.0, 1.5], dtype=float)
lam = 0.1

a_ground_truth = classo_fista_nonneg(Phi, y, lam)
print("Ground truth a*:", np.round(a_ground_truth, 6))

Ground truth a*: [0.683061 0.       1.21779 ]


## Build LCABrick circuit

In [3]:
scaffold = Scaffold()
lca_brick = scaffold.add_brick(
    LCABrick(Phi=Phi, input_signal=y, dt=1e-3, lam=lam, name="LCABrick"),
    output=True,
)
scaffold.lay_bricks()

print("Brick tag:", lca_brick.brick_tag)

Brick tag: LCABrick-0


## Run with `snn_Backend`

We intentionally use the generic simulator backend and then decode from spikes.

In [4]:
backend = snn_Backend()
backend.compile(
    scaffold=scaffold,
    compile_args={
        "record": "all",
        "ds_format": True,
    },
)

n_steps = 100000
dt = 1e-3
spike_df = backend.run(n_steps=n_steps)

print(f"Simulation done. Total simulated time: {n_steps * dt:.2f} s")
print("Spike records:", len(spike_df))

<class 'fugu.simulators.SpikingNeuralNetwork.compartments.RecurrentInhibition'>
<class 'fugu.simulators.SpikingNeuralNetwork.compartments.RecurrentInhibition'>
<class 'fugu.simulators.SpikingNeuralNetwork.compartments.RecurrentInhibition'>
Simulation done. Total simulated time: 100.00 s
Spike records: 190


## Spike Post-Processing Decoder

`snn_Backend` returns spike events only. We decode coefficients in two steps:
1. Use spike rates to estimate active support (which atoms are used).
2. Solve least-squares on that support and apply nonnegative soft-thresholding.

In [5]:
def decode_lca_from_spikes(spike_df, scaffold, lca_brick, Phi, y, lam, dt, n_steps, warmup_steps=5000):
    spikes_named = fill_results_from_graph(spike_df, scaffold, fields=["neuron_number", "name", "brick"])

    tag = lca_brick.brick_tag
    mask = spikes_named["name"].str.startswith(f"{tag}:neuron_")
    lca_spikes = spikes_named.loc[mask].copy()

    if lca_spikes.empty:
        raise RuntimeError("No LCA neuron spikes found. Increase n_steps or check setup.")
    print(lca_spikes)
    neurons = lca_spikes["name"].unique()

    lca_spikes["index"] = lca_spikes["neuron_number"].astype(int)

    tail = lca_spikes[lca_spikes["time"] >= warmup_steps]
    N = Phi.shape[1]

    counts = tail.groupby("index").size().reindex(range(N), fill_value=0).to_numpy(dtype=float)
    tail_seconds = max((n_steps - warmup_steps) * dt, 1e-12)
    rates_hz = counts / tail_seconds

    max_rate = np.max(rates_hz) if rates_hz.size else 0.0
    if max_rate <= 0.0:
        support = np.array([0], dtype=int)
    else:
        support = np.where(rates_hz >= max_rate)[0]
        if support.size == 0:
            support = np.array([int(np.argmax(rates_hz))], dtype=int)

    A = Phi[:, support]
    coef_support, *_ = np.linalg.lstsq(A, y, rcond=None)
    coef_support = np.maximum(0.0, coef_support - lam)

    a_decoded = np.zeros(N, dtype=float)
    a_decoded[support] = coef_support

    x_hat = Phi @ a_decoded
    return {
        "a_decoded": a_decoded,
        "x_hat": x_hat,
        "rates_hz": rates_hz,
        "counts": counts.astype(int),
        "support": support,
    }

In [6]:
decoded = decode_lca_from_spikes(
    spike_df=spike_df,
    scaffold=scaffold,
    lca_brick=lca_brick,
    Phi=Phi,
    y=y,
    lam=lam,
    dt=dt,
    n_steps=n_steps,
    warmup_steps=5000,
)

a_decoded = decoded["a_decoded"]
x_hat = decoded["x_hat"]
rates_hz = decoded["rates_hz"]
counts = decoded["counts"]
support = decoded["support"]

solution_error = np.linalg.norm(a_decoded - a_ground_truth)
recon_error = np.linalg.norm(x_hat - y)

summary = pd.DataFrame({
    "Neuron": [f"Neuron {i+1}" for i in range(len(a_decoded))],
    "Ground Truth a*": np.round(a_ground_truth, 6),
    "Decoded a": np.round(a_decoded, 6),
    "Abs Diff": np.round(np.abs(a_decoded - a_ground_truth), 6),
    "Spike Count (tail)": counts,
    "Spike Rate Hz (tail)": np.round(rates_hz, 4),
})

print("Support from spikes:", support)
print("Ground truth a*:", np.round(a_ground_truth, 6))
print("Decoded a:", np.round(a_decoded, 6))
print(f"Solution error ||a_decoded - a*||: {solution_error:.6e}")
print(f"Reconstruction error ||y - Phi a_decoded||: {recon_error:.6e}")

summary

        time  neuron_number                 name       brick
0      607.0            3.0  LCABrick-0:neuron_2  LCABrick-0
1      748.0            1.0  LCABrick-0:neuron_0  LCABrick-0
2     1248.0            2.0  LCABrick-0:neuron_1  LCABrick-0
3     1564.0            3.0  LCABrick-0:neuron_2  LCABrick-0
4     2292.0            1.0  LCABrick-0:neuron_0  LCABrick-0
..       ...            ...                  ...         ...
185  97369.0            3.0  LCABrick-0:neuron_2  LCABrick-0
186  98201.0            3.0  LCABrick-0:neuron_2  LCABrick-0
187  98640.0            1.0  LCABrick-0:neuron_0  LCABrick-0
188  99029.0            3.0  LCABrick-0:neuron_2  LCABrick-0
189  99819.0            3.0  LCABrick-0:neuron_2  LCABrick-0

[190 rows x 4 columns]
Support from spikes: [1]
Ground truth a*: [0.683061 0.       1.21779 ]
Decoded a: [0.       1.348618 0.      ]
Solution error ||a_decoded - a*||: 1.941226e+00
Reconstruction error ||y - Phi a_decoded||: 1.188109e+00


,Neuron,Ground Truth a*,Decoded a,Abs Diff,Spike Count (tail),Spike Rate Hz (tail)
0,Neuron 1,0.683061,0.000000,0.683061,0,0.0000
1,Neuron 2,0.000000,1.348618,1.348618,65,0.6842
2,Neuron 3,1.217790,0.000000,1.217790,0,0.0000
